#Importacion de librerias

In [1]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler
from sklearn.tree import DecisionTreeClassifier



#Preprocesamiento de datos

In [2]:
# --------------------------------------------------------------------------
# 1. CONFIGURACIÓN
# --------------------------------------------------------------------------
DATA_PATH = "https://raw.githubusercontent.com/No-Country-simulation/Hackaton_G9_Server_2_Team27/refs/heads/feature/ds-notebooks-modelos/data/processed/dataset_ml_ready_v2.csv"
TARGET_COLUMN = "categoria"
MODEL_OUTPUT_PATH = Path("model_decision_tree_pipeline.pkl")
RANDOM_STATE = 42
TEST_SIZE = 0.2

# Numéricas continuas/discretas -> RobustScaler
NUMERIC_FEATURES = ["consumo_kwh", "cantidad_equipos", "horas_alto_consumo"]

# Categóricas NOMINALES -> One-Hot
CATEGORICAL_FEATURES = ["tipo_vivienda"]

# Categóricas ORDINALES
ORDINAL_FEATURES: list[str] = []
ORDINAL_CATEGORIES: list[list[str]] = []

# Booleans -> "passthrough" directo (scikit-learn maneja bool de nativo)
BOOLEAN_FEATURES = ["uso_horario_pico"]

def build_preprocessor() -> ColumnTransformer:
    """
    Centraliza TODO el preprocesamiento en un ColumnTransformer.
    Se ajusta (fit) únicamente sobre X_train.
    """
    transformers = [
        ("num", RobustScaler(), NUMERIC_FEATURES),
        (
            "cat_nominal",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            CATEGORICAL_FEATURES,
        ),
        ("bool", "passthrough", BOOLEAN_FEATURES),
    ]

    if ORDINAL_FEATURES:
        transformers.insert(
            1,
            (
                "cat_ordinal",
                OrdinalEncoder(
                    categories=ORDINAL_CATEGORIES,
                    handle_unknown="use_encoded_value",
                    unknown_value=-1,
                ),
                ORDINAL_FEATURES,
            ),
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0,  # Asegura matriz densa en la salida
        verbose_feature_names_out=False,
    )


def build_pipeline() -> Pipeline:
    """
    Pipeline único: preprocesamiento + modelo.
    Este es el objeto que se serializa y se carga en FastAPI.
    """
    return Pipeline(
        steps=[
            ("preprocessor", build_preprocessor()),
            (
                "classifier",
                DecisionTreeClassifier(
                    criterion="gini",
                    max_depth=5,
                    min_samples_leaf=5,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )




#Entrenamiento

In [3]:

def main():
    # ----------------------------------------------------------------------
    # 2. CARGA DE DATOS
    # ----------------------------------------------------------------------
    df = pd.read_csv(DATA_PATH)

    feature_cols = (
        NUMERIC_FEATURES + CATEGORICAL_FEATURES + ORDINAL_FEATURES + BOOLEAN_FEATURES
    )
    X = df[feature_cols]
    y = df[TARGET_COLUMN]

    # ----------------------------------------------------------------------
    # 3. SPLIT TRAIN / TEST
    # ----------------------------------------------------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    # ----------------------------------------------------------------------
    # 4. ENTRENAMIENTO
    # ----------------------------------------------------------------------
    pipeline = build_pipeline()
    pipeline.fit(X_train, y_train)

    # ----------------------------------------------------------------------
    # 5. EVALUACIÓN
    # ----------------------------------------------------------------------
    y_pred = pipeline.predict(X_test)

    print("=" * 50)
    print("Accuracy")
    print("=" * 50)
    print(f"{accuracy_score(y_test, y_pred):.4f}")

    print("\n" + "=" * 50)
    print("Classification Report")
    print("=" * 50)
    print(classification_report(y_test, y_pred))

    print("=" * 50)
    print("Confusion Matrix")
    print("=" * 50)
    print(confusion_matrix(y_test, y_pred))

    # ----------------------------------------------------------------------
    # 6. EXPORTACIÓN DEL PIPELINE COMPLETO
    # ----------------------------------------------------------------------
    # Creación segura de directorios antes de guardar
    MODEL_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

    joblib.dump(pipeline, MODEL_OUTPUT_PATH)
    print(f"\nModelo guardado correctamente en: {MODEL_OUTPUT_PATH}")

    return pipeline, X_test, y_test

#Resultados

In [4]:
if __name__ == "__main__":
    main()

Accuracy
0.8172

Classification Report
              precision    recall  f1-score   support

           0       0.90      0.77      0.83       655
           1       0.68      0.84      0.75       657
           2       0.92      0.84      0.88       668

    accuracy                           0.82      1980
   macro avg       0.83      0.82      0.82      1980
weighted avg       0.84      0.82      0.82      1980

Confusion Matrix
[[503 151   1]
 [ 54 555  48]
 [  0 108 560]]

Modelo guardado correctamente en: model_decision_tree_pipeline.pkl


#Conclusion del Modelo

Conclusión del Modelo

Desempeño General del Modelo

El modelo de Árbol de Decisión alcanza un rendimiento sólido y balanceado, logrando una exactitud general (accuracy) del 81.72% sobre un total de 1,980 instancias de prueba. Dado que las clases de evaluación se encuentran equilibradas en el conjunto de test (aproximadamente 660 observaciones por clase), el f1-score macro de 0.82 confirma que el modelo mantiene un buen comportamiento predictivo de forma homogénea.

Análisis por Clase de Clasificación

Clase 0 (Alta Precisión):Alcanza una precisión del 90% y un F1-Score de 0.83. El modelo es muy confiable cuando predice esta clase (pocos falsos positivos), aunque tiende a confundir un 23% de estos casos clasificándolos incorrectamente como Clase 1.

Clase 1 (Alta Sensibilidad / Recall):Registra la mayor tasa de detección con un recall del 84%, aunque su precisión desciende al 68%. Esto indica que la Clase 1 actúa como la zona intermedia del modelo, absorbiendo falsos positivos provenientes tanto de la Clase 0 (151 casos) como de la Clase 2 (108 casos).

Clase 2 (Mejor Desempeño Global):Es la clase mejor clasificada por el árbol, con una precisión del 92%, un recall del 84% y el F1-Score más alto (0.88). Además, muestra una separación casi perfecta con la Clase 0 (0 confusiones cruzadas)

Matriz de Confusión y Comportamiento de Errores

La matriz de confusión refleja una estructura ordinal o de gradiente claro:Los errores ocurren únicamente entre clases adyacentes (Clase $0 \leftrightarrow 1$ y Clase $1 \leftrightarrow 2$).

Existe un solapamiento muy bajo entre los extremos: solo 1 caso de Clase 0 fue predicho como Clase 2, y 0 casos de Clase 2 fueron predichos como Clase 0.

Veredicto y Recomendaciones

Viabilidad de Despliegue: El pipeline guardado en model_decision_tree_pipeline.pkl está listo para producción como una línea base sólida y explicable.

Siguiente Paso Sólido: Para reducir el ruido de clasificación en la Clase 1, se recomienda evaluar la restricción de profundidad (max_depth) para evitar overfitting, o comparar los resultados contra un algoritmo de ensamble como Random Forest o XGBoost, los cuales suelen suavizar las fronteras de decisión en clases intermedias.